In [15]:
import torch
import torch.optim
import numpy as np
import scipy
import torchvision


In [ ]:
testTensor = torch.ones((5,5,5)).to()
torch.cuda.is_available
print(torch.cuda.is_available())
print(torch.cuda.current_device)
print(torch.cuda.get_device_name(0))

: 

In [ ]:
class testNN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        #for input image size imageSize, 1 per batch
        imageSize = 28
        poolKernel = 2
        convoKernel = 5
        self.convo1 = torch.nn.Conv2d(1,6,convoKernel)
        self.Max2 = torch.nn.MaxPool2d(poolKernel)
        self.convo3 = torch.nn.Conv2d(6,16,convoKernel)
        self.Max4 = torch.nn.MaxPool2d(poolKernel)
        self.imageAfterPP = (((imageSize-(convoKernel-1))/2)-((convoKernel-1)))/2
        self.linear5 = torch.nn.Linear(int(16*self.imageAfterPP*self.imageAfterPP),120)
        self.linear6 = torch.nn.Linear(120,84)
        self.linear7 = torch.nn.Linear(84,10)

    def forward(self,input):
        input = self.convo1(input)
        input = self.Max2(input)
        input = self.convo3(input)
        input = self.Max4(input)
        input = torch.flatten(input,1)
        input = torch.nn.functional.relu(self.linear5(input))
        input = torch.nn.functional.relu(self.linear6(input))
        input = torch.nn.functional.relu(self.linear7(input))
        return input

testNet = testNN()
print(testNet)

: 

In [ ]:
import torch
torch.cuda.is_available()

: 

In [ ]:
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self,listID,label,transform=None):
        self.listID = listID.cuda()
        self.label = label.cuda()
        self.transform = transform
        
    def __len__(self):
        return len(self.listID)
    
    def __getitem__(self,idx):#generate pair of label,data
        if self.transform:
            print("im too lazy to implement support for transforms lmaooooo")

        return torch.unsqueeze(self.listID[idx],0),self.label[idx]
        

: 

In [ ]:
import pandas
import numpy as np
import idx2numpy

trainFile = r".\train-images.idx3-ubyte"
labelFile = r".\train-labels.idx1-ubyte"
# simple import (took me half an hour)
train = torch.from_numpy(idx2numpy.convert_from_file(trainFile)).float()
label = torch.from_numpy(idx2numpy.convert_from_file(labelFile)).long()



: 

In [ ]:
batchSize = 4
testNet.cuda()
trainData = CustomDataset(train,label)
trainData = torch.utils.data.random_split(trainData,[0.7,0.3])
trainLoader = torch.utils.data.DataLoader(trainData[0],batch_size= batchSize,shuffle=False, num_workers = 0 )
testLoader = torch.utils.data.DataLoader(trainData[1], batch_size=batchSize, shuffle=False, num_workers=0)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(testNet.parameters(),lr = 0.001 ) 

: 

In [ ]:
epochs = 1
running_loss = 0
print(path)
for i in range(epochs):
    for idx, idkWhatToNameIt in enumerate(trainLoader,0):
        data,label = idkWhatToNameIt

        optimizer.zero_grad()
        # forward + backward + optimize
        outputs = testNet(data)
        loss = criterion(outputs, label)
        loss.backward()
        optimizer.step()

        # print statistics  
        running_loss += loss.item()
        if idx % 200 == 199:    # print every 2000 mini-batches
            print(f'[{epochs + 1}, {i + 1:5d}] loss: {running_loss / 200:.3f}')
            running_loss = 0.0
print("fasulid")

: 

In [ ]:
import matplotlib.pyplot as plt
enum = enumerate(testLoader)
for i in range(3):
    fig = plt.figure(figsize=(10, 7))
    idx,data = next(enum)
    input,label = data
    optimizer.zero_grad()
    output = testNet(input)
    for i in range(4):
        npimg = input.cpu().data.numpy()
        fig.add_subplot(1,8, i+1)
        plt.imshow(npimg[i,0])
        plt.title(np.argmax(output[i].cpu().data.numpy()))
    plt.show()
        

: 